# Academy of Bards: live check

Notebook version of `fanfic-verify-academy`. **Makes 7 requests**: robots.txt, the author index, and 5 author pages chosen for their markup variety. All go through the project fetcher, so the 5–10 s spacing, robots.txt rules and backoff apply.

The pages are saved to the data folder. Re-running reads them from the saved copies, and only robots.txt is requested again.

In [ ]:
import os
import pathlib
import sys

# Find the repository root (the folder holding pyproject.toml) whether this
# notebook was opened from notebooks/ or from the root, and work from there,
# so saved pages and outputs always go to <repo>/data/ (or FANFIC_DATA_DIR).
REPO_ROOT = pathlib.Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Open this notebook from inside the repository folder.")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # works without `pip install -e .`
REPO_ROOT

## Settings

`CONTACT_EMAIL` is sent with every request (in the User-Agent) so site operators can reach whoever is crawling. Use your own address. Leave it blank to be asked the first time a download is needed, or set the `FANFIC_CONTACT_EMAIL` environment variable instead.

In [ ]:
CONTACT_EMAIL = ""  # your email address

## robots.txt (network)

Requested once per run and never saved. The fetcher refuses to crawl if it can't be read.

In [ ]:
from fanfic_scraper.fetch import ACADEMY, build_default_fetcher

fetcher = build_default_fetcher(ACADEMY, contact_email=CONTACT_EMAIL or None)
robots = fetcher.robots_for(ACADEMY.host)
print(f"robots.txt for {ACADEMY.host}:")
print(f"  disallow_prefixes = {robots.disallow_prefixes}")
print(f"  crawl_delay       = {robots.crawl_delay}")

## Pages (network the first time, saved copies after)

In [ ]:
from fanfic_scraper.discover.academy import discover_academy

index_page, entries, sample_pages = discover_academy(fetcher)
print(f"{len(entries)} unique author URLs found in the index")
for page in sample_pages:
    print(f"  {page.url}: status={page.status} bytes={len(page.body)}")